In [1]:
import pandas as pd
import numpy as np

# 1. Load the data we scraped
# (Assuming you ran the deep scraper and it saved the CSV)
df = pd.read_csv('deep_lagos_rent_data.csv')

print(f"Original dataset shape: {df.shape}")

# 2. Clean the Amenities column (fill empty spaces so the code doesn't crash)
df['Amenities'] = df['Amenities'].fillna('')

# 3. THE FEATURE EXPLOSION (Multi-Label Binarization)
# This single line of code reads the comma-separated text, finds every unique 
# feature across the entire dataset, and creates a new column for it with a 1 or 0.
amenities_df = df['Amenities'].str.get_dummies(sep=', ')

# 4. Join the new binary columns back to our main dataset
df = pd.concat([df, amenities_df], axis=1)

# 5. Drop the old messy text column since we extracted the math from it
df = df.drop('Amenities', axis=1)

print(f"New dataset shape after feature explosion: {df.shape}")

# Display the first 5 rows to visually confirm the new columns
df.head()

Original dataset shape: (3, 6)
New dataset shape after feature explosion: (3, 5)


,URL,Title,Is_Serviced,Is_Furnished,Description_Snippet
0,https://www.propertypro.ng/property/2-bedroom-...,Rent Furnished 2 Bedroom Maisonette in Lekki L...,1,1,"PAY & MOVE IN (agency, legal, refundable cuati..."
1,https://www.propertypro.ng/property/4-bedroom-...,Rent Newly Built 4 Bedroom Terrace Duplex in L...,0,0,Features: \n - All Rooms Ensuite \n ...
2,https://www.propertypro.ng/property/4-bedroom-...,Rent Newly Built 4 Semi Bedroom Detached Duple...,0,0,4BEDS || AJAH LEKKI ???? ||N180M \n \nG...


In [2]:
import pandas as pd
import numpy as np
import re

# 1. Load the newly populated data
df = pd.read_csv('deep_lagos_rent_data.csv')

print(f"Starting Shape: {df.shape}")

# ---------------------------------------------------------
# STEP A: CLEANING THE PRICES
# ---------------------------------------------------------
def clean_price(price_string):
    if pd.isna(price_string) or price_string == 'N/A':
        return np.nan
    
    # Regex: Remove everything that is NOT a digit (strips ₦, commas, and text)
    clean_string = re.sub(r'[^\d]', '', str(price_string))
    
    try:
        return float(clean_string)
    except:
        return np.nan

df['Price_Clean'] = df['Price'].apply(clean_price)

# Drop any rows where the price is missing (we can't train ML on missing prices)
df = df.dropna(subset=['Price_Clean'])

# Drop the old dirty 'Price' column
df = df.drop('Price', axis=1) 


# ---------------------------------------------------------
# STEP B: MULTI-LABEL BINARIZATION (Exploding the Amenities)
# ---------------------------------------------------------
df['Amenities'] = df['Amenities'].fillna('')

# This creates a massive matrix of 1s and 0s for every unique feature found
amenities_df = df['Amenities'].str.get_dummies(sep=', ')

# Join the new binary columns back to our main dataset
df = pd.concat([df, amenities_df], axis=1)

# Drop the old text column
df = df.drop('Amenities', axis=1)

print(f"Final ML-Ready Dataset Shape: {df.shape}")
print("-" * 50)

# Display the final cleaned, algorithmic-ready data
df.head()

Starting Shape: (22, 5)
Final ML-Ready Dataset Shape: (22, 29)
--------------------------------------------------


,Location,Is_Serviced,Is_Furnished,Price_Clean,24 Hours Security,24 hours Electricity,All Room Ensuit,Big Compound,Boys Quater,C of O,...,Mosques Nearby,POP Ceiling,Parking Space,Restaurants Nearby,Security,Security doors,Street Lights,Supermarket Nearby,Swimming Pool,Water Treatment
0,Lekki Phase 1 Lekki Lagos,1,0,10000000.0,1,1,1,1,0,0,...,0,1,1,1,1,1,1,1,1,1
1,Ajah Lekki Lagos,0,0,7000000.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Lekki County Lekki Lagos,0,0,11000000.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Ajah Lekki Lagos,0,0,12000000.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Lekki Phase 1 Lekki Lagos,1,0,12000000.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
# 1. One-Hot Encode the Location column to handle Geographical Bias
location_dummies = pd.get_dummies(df['Location'], prefix='Loc')

# 2. Join the new location matrix to our dataset
df = pd.concat([df, location_dummies], axis=1)

# 3. Drop the old text columns so the dataset is 100% mathematical
df = df.drop(['Location', 'URL', 'Title', 'Description_Snippet'], axis=1)

# 4. Define our 'Target' (y) and our 'Features' (X)
# y is what we want to predict (Price)
y = df['Price_Clean']

# X is the data we use to predict the price (Everything EXCEPT the price)
X = df.drop('Price_Clean', axis=1)

print(f"Features matrix (X) shape: {X.shape}")
print(f"Target vector (y) shape: {y.shape}")
print("\nSuccess! The dataset is now a pure mathematical matrix ready for Machine Learning.")

# View the final features
X.head()

KeyError: "['URL', 'Title', 'Description_Snippet'] not found in axis"

In [4]:
# 1. One-Hot Encode the Location column to handle Geographical Bias
location_dummies = pd.get_dummies(df['Location'], prefix='Loc')

# 2. Join the new location matrix to our dataset
df = pd.concat([df, location_dummies], axis=1)

# 3. Drop the old text columns so the dataset is 100% mathematical
# We drop 'Location'. 
# Note: If you haven't processed 'Amenities' into binary columns yet, 
# you will need to drop that too before passing it to the ML models.
columns_to_drop = ['Location']
if 'Amenities' in df.columns:
    columns_to_drop.append('Amenities') # Remove this if you've already applied Multi-Label Binarization

df = df.drop(columns_to_drop, axis=1)

# 4. Define our 'Target' (y) and our 'Features' (X)
# Double-check that your Regex cleaning step ran successfully to create 'Price_Clean'
if 'Price_Clean' not in df.columns:
    raise KeyError("'Price_Clean' is missing. Make sure you run the Regex cell to clean the 'Price' column before running this block.")

# y is what we want to predict (Price)
y = df['Price_Clean']

# X is the data we use to predict the price (Everything EXCEPT the price and the old Price column)
if 'Price' in df.columns:
    df = df.drop('Price', axis=1) # Drop the original raw text price if it's still hanging around

X = df.drop('Price_Clean', axis=1)

print(f"Features matrix (X) shape: {X.shape}")
print(f"Target vector (y) shape: {y.shape}")
print("\nSuccess! The dataset is now a pure mathematical matrix ready for Machine Learning.")

# View the final features
X.head()

Features matrix (X) shape: (22, 47)
Target vector (y) shape: (22,)

Success! The dataset is now a pure mathematical matrix ready for Machine Learning.


,Is_Serviced,Is_Furnished,24 Hours Security,24 hours Electricity,All Room Ensuit,Big Compound,Boys Quater,C of O,CCTV Cameras,Child Care,...,Loc_Ajah Lekki Lagos,Loc_Chevron Alternative Route Lekki Lagos,Loc_Ikota Lekki Lagos,Loc_Lekki County Lekki Lagos,Loc_Lekki Lagos,Loc_Lekki Phase 1 Lekki Lagos,Loc_Ologolo Lekki Lagos,Loc_Orchid Lekki Lagos,Loc_Parkview Estate Ikoyi Lagos,Loc_Vgc Lekki Lagos
0,1,0,1,1,1,1,0,0,1,1,...,False,False,False,False,False,True,False,False,False,False
1,0,0,0,0,0,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False
2,0,0,0,0,0,0,0,0,0,0,...,False,False,False,True,False,False,False,False,False,False
3,0,0,0,0,0,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False
4,1,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
